# Moduł 9: Sieci konwolucyjne (CNN)

CNN to architektura sieci neuronowych zaprojektowana do przetwarzania **obrazów** (i innych danych o strukturze przestrzennej).

## Spis treści
1. Konwolucja — intuicja
2. Warstwy CNN
3. Pooling
4. Architektura CNN
5. CNN na MNIST w PyTorch
6. Transfer learning
7. Detekcja i segmentacja (przegląd)
8. **Zadania**

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

## 1. Konwolucja — intuicja i matematyka

**Konwolucja** to operacja, w której mały filtr (kernel) "przesuwa się" po obrazie i oblicza iloczyn skalarny z fragmentem obrazu.

### Operacja konwolucji 2D (właściwie: korelacja krzyżowa)

Dla wejścia $I$ i kernela $K$ o rozmiarze $k \times k$:

$$(I * K)(i, j) = \sum_{m=0}^{k-1} \sum_{n=0}^{k-1} I(i+m, j+n) \cdot K(m, n)$$

W PyTorch (`Conv2d`) to technicznie **korelacja krzyżowa** (cross-correlation), nie konwolucja matematyczna (która odwraca kernel). W praktyce nie ma to znaczenia, bo kernel jest uczony.

### Konwolucja wielokanałowa

Dla wejścia z $C_{\text{in}}$ kanałami i $C_{\text{out}}$ filtrami:

$$(Y)_j(i_1, i_2) = b_j + \sum_{c=1}^{C_{\text{in}}} \sum_{m,n} W_{j,c}(m, n) \cdot X_c(i_1+m, i_2+n)$$

gdzie $j = 1, \ldots, C_{\text{out}}$ — indeks filtra (kanału wyjściowego).

### Dlaczego konwolucja, a nie MLP?

| Własność | MLP | CNN |
|----------|-----|-----|
| Współdzielenie wag | ❌ Każdy piksel → osobna waga | ✅ Ten sam filtr na całym obrazie |
| Parametry | 28×28 = 784 wejść → dużo | Filtr 3×3 = 9 parametrów |
| Lokalność | ❌ Ignoruje sąsiedztwo | ✅ Lokalne wzorce (krawędzie, tekstury) |
| Translacja | ❌ Nie jest invariant | ✅ Invariant (z poolingiem) |

### Trzy kluczowe własności CNN
1. **Rzadkie połączenia** (sparse connectivity) — każdy neuron łączy się tylko z lokalnym regionem
2. **Współdzielenie wag** (weight sharing) — ten sam filtr na całym obrazie
3. **Ekwiwariantność translacji** — przesunięty wzór → przesunięta odpowiedź

### Hierarchia cech

```
Warstwa 1: krawędzie, gradienty
Warstwa 2: tekstury, wzorce
Warstwa 3: części obiektów (oczy, koła)
Warstwa 4+: całe obiekty (twarz, samochód)
```

### Kernel (filtr)
```
Wykrywanie krawędzi pionowych:    Wykrywanie krawędzi poziomych:
[-1  0  1]                        [-1 -1 -1]
[-1  0  1]                        [ 0  0  0]
[-1  0  1]                        [ 1  1  1]
```

In [ ]:
# Demonstracja konwolucji na obrazie
from torchvision import datasets, transforms
import torch.nn.functional as F

# Załaduj jeden obraz MNIST
mnist = datasets.MNIST('./data', train=True, download=True, transform=transforms.ToTensor())
img, label = mnist[0]
print(f"Obraz: {img.shape}, cyfra: {label}")

# Ręczne filtry
kernels = {
    "Vertical edge": torch.tensor([[-1., 0., 1.], [-1., 0., 1.], [-1., 0., 1.]]),
    "Horizontal edge": torch.tensor([[-1., -1., -1.], [0., 0., 0.], [1., 1., 1.]]),
    "Sharpen": torch.tensor([[0., -1., 0.], [-1., 5., -1.], [0., -1., 0.]]),
    "Blur": torch.ones(3, 3) / 9
}

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
axes[0].imshow(img.squeeze(), cmap='gray')
axes[0].set_title("Oryginał")

for i, (name, kernel) in enumerate(kernels.items()):
    k = kernel.unsqueeze(0).unsqueeze(0)  # (1, 1, 3, 3)
    result = F.conv2d(img.unsqueeze(0), k, padding=1)
    axes[i+1].imshow(result.squeeze().detach(), cmap='gray')
    axes[i+1].set_title(name)

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Warstwy CNN — parametry

### `nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)`

- **in_channels** ($C_{\text{in}}$): kanały wejściowe (1 = grayscale, 3 = RGB)
- **out_channels** ($C_{\text{out}}$): liczba filtrów (= liczba map cech na wyjściu)
- **kernel_size** ($k$): rozmiar filtra (3 lub 5 to typowe)
- **stride** ($s$): krok przesuwania filtra (domyślnie 1)
- **padding** ($p$): dodatkowe piksele wokół obrazu (zachowanie rozmiaru)

### Obliczanie rozmiaru wyjścia

$$H_{\text{out}} = \left\lfloor\frac{H_{\text{in}} - k + 2p}{s}\right\rfloor + 1$$

Analogicznie dla szerokości $W_{\text{out}}$.

Przykład: $H_{\text{in}}=28$, $k=3$, $p=1$, $s=1$:

$$\left\lfloor\frac{28 - 3 + 2}{1}\right\rfloor + 1 = 28 \text{ (zachowany rozmiar!)}$$

### Zachowanie rozmiaru — reguła

Dla kernela $k$ (nieparzyste) i stride=1, padding $p = \lfloor k/2 \rfloor$ zachowuje rozmiar:
- kernel 3×3 → padding 1
- kernel 5×5 → padding 2
- kernel 7×7 → padding 3

### Liczba parametrów warstwy konwolucyjnej

$$\text{params} = C_{\text{out}} \times (C_{\text{in}} \times k \times k + 1)$$

$+1$ za bias na każdy filtr.

Przykład: `Conv2d(3, 64, 3)` → $64 \times (3 \times 3 \times 3 + 1) = 64 \times 28 = 1{,}792$ parametrów

### Dilated Convolution (Atrous)

Rozszerzony kernel z "dziurami" — zwiększa **pole receptywne** bez zwiększania parametrów:

$$H_{\text{out}} = \left\lfloor\frac{H_{\text{in}} - d(k-1) - 1 + 2p}{s}\right\rfloor + 1$$

gdzie $d$ to dilation rate.

### Pole receptywne (Receptive Field)

Pole receptywne neuronu w warstwie $l$ — ile pikseli wejściowych na niego wpływa:

$$r_l = r_{l-1} + (k_l - 1) \cdot \prod_{i=1}^{l-1} s_i$$

Głębsze warstwy "widzą" większy fragment obrazu wejściowego.

### Konwolucja 1×1

Specjalny przypadek — zmienia **liczbę kanałów** bez zmiany rozmiaru przestrzennego:
- Redukcja wymiarów: Conv2d(256, 64, 1)  
- Użycie: bottleneck w ResNet, Inception

## 3. Pooling

**Pooling** zmniejsza rozmiar map cech, zachowując najważniejsze informacje.

### Max Pooling 2×2
```
[1 3 | 5 2]        [3 | 5]
[4 2 | 1 3]  →     [7 | 3]
----------- 
[7 1 | 2 3]
[5 6 | 0 1]
```
Bierze **max** z każdego regionu 2×2. Rozmiar maleje 2×.

### Rodzaje poolingu

| Typ | Operacja | Zastosowanie |
|-----|----------|-------------|
| **Max Pooling** | $\max$ z regionu | Najpopularniejszy, zachowuje silne cechy |
| **Average Pooling** | $\text{mean}$ z regionu | Gładszy, mniej agresywny |
| **Global Average Pooling (GAP)** | $\text{mean}$ z **całej** mapy cech | Zamiennik Flatten+FC |

### Rozmiar po poolingu

$$H_{\text{out}} = \left\lfloor\frac{H_{\text{in}}}{k_{\text{pool}}}\right\rfloor$$

Typowo $k_{\text{pool}} = 2$ → rozmiar zmniejsza się 2× w każdym wymiarze (4× total pikseli).

### Pooling ma 0 parametrów!
Pooling to **operacja deterministyczna** — nie ma wag do uczenia. Zmniejsza liczbę parametrów w następnych warstwach.

### Stride Convolution jako alternatywa

Zamiast poolingu, można zmniejszyć rozmiar konwolucją ze stride > 1:
- `Conv2d(32, 64, 3, stride=2, padding=1)` — zmniejsza rozmiar 2× **i** uczy się jak redukować
- Używane w nowoczesnych architekturach (ResNet, EfficientNet)

## 4. Typowa architektura CNN

```
Input (1×28×28)
  → Conv2d(1→32, 3×3, padding=1)  →  ReLU  →  MaxPool(2×2)   # 32×14×14
  → Conv2d(32→64, 3×3, padding=1) →  ReLU  →  MaxPool(2×2)   # 64×7×7
  → Flatten()                                                   # 3136
  → Linear(3136→128) → ReLU → Dropout
  → Linear(128→10)                                              # 10 klas
```

### Obliczenie parametrów krok po kroku

| Warstwa | Rozmiar | Parametry |
|---------|---------|-----------|
| Conv2d(1, 32, 3) | 32×28×28 | $32 \times (1 \times 3 \times 3 + 1) = 320$ |
| Conv2d(32, 64, 3) | 64×14×14 | $64 \times (32 \times 3 \times 3 + 1) = 18{,}496$ |
| Linear(3136, 128) | 128 | $3136 \times 128 + 128 = 401{,}536$ |
| Linear(128, 10) | 10 | $128 \times 10 + 10 = 1{,}290$ |
| **Razem** | | **~421,642** |

> Uwaga: większość parametrów jest w **warstwie FC**, nie w konwolucyjnych!

### Znane architektury CNN

| Architektura | Rok | Kluczowa innowacja | Top-5 ImageNet |
|-------------|------|-------|------|
| **LeNet-5** | 1998 | Pierwsza CNN | — |
| **AlexNet** | 2012 | ReLU, Dropout, GPU | 15.3% |
| **VGG** | 2014 | Małe filtry 3×3 | 7.3% |
| **GoogLeNet/Inception** | 2014 | Moduły Inception (równoległe filtry) | 6.7% |
| **ResNet** | 2015 | Połączenia rezydualne (skip connections) | 3.6% |
| **EfficientNet** | 2019 | NAS + compound scaling | 2.9% |
| **ViT** | 2020 | Transformery zamiast konwolucji | 1.8% |

### ResNet — Skip Connections

**Problem:** Bardzo głębokie sieci (>20 warstw) trudno trenować — degradacja gradientu.

**Rozwiązanie:** Połączenia rezydualne (skip/shortcut connections):

$$\mathbf{y} = \mathcal{F}(\mathbf{x}, \{W_i\}) + \mathbf{x}$$

Sieć uczy się **residuum** $\mathcal{F}(\mathbf{x}) = H(\mathbf{x}) - \mathbf{x}$ zamiast całej transformacji $H(\mathbf{x})$.

**Dlaczego to pomaga?** Gradient przepływa bezpośrednio przez skip connection: $\frac{\partial \mathcal{L}}{\partial \mathbf{x}} = \frac{\partial \mathcal{L}}{\partial \mathbf{y}}(1 + \frac{\partial \mathcal{F}}{\partial \mathbf{x}})$ → nie zanika!

## 5. CNN na MNIST w PyTorch

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),   # 1×28×28 → 32×28×28
            nn.ReLU(),
            nn.MaxPool2d(2),                               # → 32×14×14
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # → 64×14×14
            nn.ReLU(),
            nn.MaxPool2d(2),                               # → 64×7×7
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),                                  # → 3136
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 10)
        )
    
    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

model_cnn = CNN()
print(model_cnn)
total_params = sum(p.numel() for p in model_cnn.parameters())
print(f"\nParametrów: {total_params:,}")
# Porównaj z MLP: CNN ma DUŻO mniej parametrów!

In [ ]:
# Trening CNN
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_data = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST('./data', train=False, transform=transform)
train_set, val_set = random_split(train_data, [50000, 10000])

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = DataLoader(val_set, batch_size=64)
test_loader = DataLoader(test_data, batch_size=64)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_cnn = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_cnn.parameters(), lr=0.001)

# Reużyj train_epoch i evaluate z Modułu 8
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        out = model(X_b)
        loss = criterion(out, y_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_b.size(0)
        correct += (out.argmax(1) == y_b).sum().item()
        total += X_b.size(0)
    return total_loss / total, correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for X_b, y_b in loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            out = model(X_b)
            loss = criterion(out, y_b)
            total_loss += loss.item() * X_b.size(0)
            correct += (out.argmax(1) == y_b).sum().item()
            total += X_b.size(0)
    return total_loss / total, correct / total

for epoch in range(5):
    t_loss, t_acc = train_epoch(model_cnn, train_loader, criterion, optimizer, device)
    v_loss, v_acc = evaluate(model_cnn, val_loader, criterion, device)
    print(f"Epoch {epoch+1}: train_acc={t_acc:.4f}, val_acc={v_acc:.4f}")

test_loss, test_acc = evaluate(model_cnn, test_loader, criterion, device)
print(f"\n>>> Test accuracy: {test_acc:.4f} (CNN vs ~98% MLP)")

## 6. Transfer Learning

**Idea:** Weź model **wytrenowany** na dużym zbiorze (np. ImageNet — 1.2M obrazów, 1000 klas) i dostosuj do swojego zadania.

### Dlaczego transfer learning działa?

Wczesne warstwy CNN uczą się **ogólnych cech** wizualnych (krawędzie, tekstury, kolory) — przydatnych w każdym zadaniu wizualnym. Późniejsze warstwy uczą się cech coraz bardziej **specyficznych** dla zadania.

### Podejścia

| Podejście | Co robimy | Kiedy |
|-----------|-----------|-------|
| **Feature extraction** | Zamrażamy conv, trenujemy tylko FC | Mały dataset, podobna domena |
| **Fine-tuning** | Odmrażamy kilka ostatnich warstw conv | Średni dataset |
| **Full fine-tuning** | Trenujemy cały model (mały LR) | Duży dataset, inna domena |

### Schemat fine-tuningu

1. Załaduj pretrenowany model (np. ResNet18)
2. Zamroź wagi: `param.requires_grad = False`
3. Zamień głowę klasyfikacji: `model.fc = nn.Linear(512, num_classes)`  
4. Trenuj z **małym** learning rate ($10^{-4}$ lub mniej)
5. Opcjonalnie: odmróź ostatnie warstwy conv po kilku epokach

### Data Augmentation — sztuczne powiększanie zbioru

Losowe transformacje obrazów podczas treningu:
- Obroty, odbicia, skalowanie
- Color jitter (zmiana jasności/kontrastu)
- Random crop
- Cutout / CutMix / MixUp

$$\text{Augmented sample:} \quad \tilde{x} = T(x), \quad T \sim \text{RandomTransform}$$

In [ ]:
# Transfer learning — schemat (bez pełnego treningu, bo potrzebuje dużo czasu)
# Wczytanie pretrenowanego ResNet18
resnet = models.resnet18(pretrained=True)

# Zamrożenie wszystkich warstw
for param in resnet.parameters():
    param.requires_grad = False

# Zamiana ostatniej warstwy (FC) na naszą
num_classes = 10  # np. CIFAR-10
resnet.fc = nn.Linear(resnet.fc.in_features, num_classes)
# Tylko resnet.fc będzie trenowane!

trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
total = sum(p.numel() for p in resnet.parameters())
print(f"Trenowalne: {trainable:,} / {total:,} ({trainable/total*100:.1f}%)")

## 7. Detekcja i segmentacja — przegląd

| Zadanie | Opis | Wyjście | Algorytmy |
|---------|------|---------|-----------|
| **Klasyfikacja** | Co jest na obrazie? | 1 klasa | CNN, ResNet, ViT |
| **Detekcja** | Gdzie jest + co? | Bounding boxy + klasy | R-CNN, YOLO, SSD |
| **Segmentacja semantyczna** | Etykieta na każdy piksel | Mapa klas | U-Net, DeepLab |
| **Segmentacja instancji** | Maska per obiekt | Maski + klasy | Mask R-CNN |

### IoU (Intersection over Union)

$$\text{IoU} = \frac{|A \cap B|}{|A \cup B|} = \frac{\text{Area of overlap}}{\text{Area of union}}$$

Progi typowe: IoU > 0.5 (PASCAL VOC), IoU > 0.5:0.95 (COCO)

### mAP (mean Average Precision)

1. Dla każdej klasy oblicz krzywą Precision-Recall
2. AP = pole pod krzywą P-R
3. mAP = średnia AP po wszystkich klasach

### YOLO (You Only Look Once) — idea

1. Dzieli obraz na siatkę $S \times S$
2. Każda komórka przewiduje $B$ bounding boxów: $(x, y, w, h, \text{confidence})$
3. Każda komórka przewiduje rozkład klas
4. **Jeden forward pass** → detekcja w czasie rzeczywistym!

### U-Net — segmentacja

Architektura encoder-decoder z **skip connections** między odpowiadającymi warstwami:

```
Encoder (downsampling)          Decoder (upsampling)
Conv → Pool → Conv → Pool  →   UpConv → Concat → Conv → UpConv → Concat → Conv
     ↘                    ↗
      Bottleneck
```

Skip connections z encodera do decodera pozwalają zachować szczegóły przestrzenne.

---
---
# 🏋️ ZADANIA

---

### Zadanie 1: Wizualizacja filtrów (łatwe)

Po wytrenowaniu CNN na MNIST, zwizualizuj nauczone filtry z pierwszej warstwy konwolucyjnej.

Podpowiedź: `model.conv_layers[0].weight.data`

In [ ]:
# Zadanie 1
# TWÓJ KOD TUTAJ

### Zadanie 2: CNN na CIFAR-10 (trudne)

CIFAR-10: 32×32 kolorowe (3 kanały) obrazy, 10 klas (samolot, auto, ptak...).

1. Załaduj CIFAR-10 z data augmentation (losowe odbicia, przycinanie)
2. Zaprojektuj CNN: ≥3 warstwy conv, BatchNorm, Dropout
3. Trenuj z Adam + scheduler
4. Cel: ≥75% test accuracy

In [ ]:
# Zadanie 2: CIFAR-10
# TWÓJ KOD TUTAJ

### Zadanie 3: Oblicz rozmiar wyjścia (łatwe)

Oblicz ręcznie (na papierze lub w komórce) rozmiar wyjścia po każdej warstwie:

```
Input: 3×32×32
Conv2d(3, 16, kernel=5, stride=1, padding=2)
MaxPool2d(2)
Conv2d(16, 32, kernel=3, stride=1, padding=0)
MaxPool2d(2)
Conv2d(32, 64, kernel=3, stride=2, padding=1)
```

Sprawdź odpowiedź, przepuszczając losowy tensor przez warstwy.

In [ ]:
# Zadanie 3
# TWÓJ KOD TUTAJ

### Zadanie 4: Autoenkoder konwolucyjny (trudne)

Zbuduj **konwolucyjny autoenkoder** na MNIST:
- **Enkoder:** Conv → ReLU → MaxPool → Conv → ReLU → MaxPool (kompresja)
- **Dekoder:** ConvTranspose → ReLU → ConvTranspose → Sigmoid (rekonstrukcja)
- Loss: MSE (piksel po pikselu)

Pokaż: oryginalny obraz vs rekonstrukcja.

In [ ]:
# Zadanie 4: Autoenkoder
# TWÓJ KOD TUTAJ

---

## 🏆 Dodatek OAI: Ćwiczenia w stylu olimpiadowym

### Kontekst olimpiadowy
CNN i wizja komputerowa to **najczęstsza kategoria** na OAI (~10 z 24 zadań!):

- **I OAI:** Ataki adwersarialne (FGSM na CNN), kwantyzacja kolorów, śledzenie obiektów
- **II OAI, Etap 1:** Maszynka do liczenia monet (detekcja/segmentacja), noisy labels (SmallMobileNet)
- **II OAI, Etap 2:** Rozkład nienormalny (denoising + klasyfikacja obrazów)
- **II OAI, Finał:** Inpainting, prototypy danych (CIFAR-100 embeddingi)
- **III OAI:** Filtry konwolucyjne, klasyfikacja wieloetykietowa, segmentacja multispektralna

### Ćwiczenie OAI-9A: Atak FGSM (Fast Gradient Sign Method)

Wytrenuj CNN na Fashion-MNIST. Potem wygeneruj perturbowane obrazy metodą FGSM i sprawdź, jak spada accuracy.

### Ćwiczenie OAI-9B: Klasyfikacja wieloetykietowa (multi-label)

Stwórz CNN z wyjściem sigmoid (nie softmax!) do zadania multi-label na syntetycznych danych. Użyj BCELoss i mierz F1 macro.

### Ćwiczenie OAI-9C: Denoising Autoencoder

Dodaj szum gaussowski do obrazów MNIST, wytrenuj autoencoder (encoder CNN + decoder deconv), zmierz PSNR rekonstrukcji.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

# === OAI-9A: Atak FGSM ===
print("=== FGSM Attack (wzorzec: Ataki adwersarialne, I OAI) ===")

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128), nn.ReLU(),
            nn.Linear(128, 10)
        )
    
    def forward(self, x):
        return self.classifier(self.features(x))

def fgsm_attack(model, images, labels, epsilon=0.1):
    """Fast Gradient Sign Method — perturbacja obrazów."""
    images.requires_grad = True
    outputs = model(images)
    loss = nn.CrossEntropyLoss()(outputs, labels)
    model.zero_grad()
    loss.backward()
    
    # Perturbacja w kierunku gradientu
    perturbed = images + epsilon * images.grad.sign()
    perturbed = torch.clamp(perturbed, 0, 1)
    return perturbed

# Demonstracja na losowym mini-batchu
model = SimpleCNN()
dummy_images = torch.rand(8, 1, 28, 28)
dummy_labels = torch.randint(0, 10, (8,))

# Predykcje oryginalne
with torch.no_grad():
    orig_preds = model(dummy_images).argmax(dim=1)

# Atak FGSM
model.eval()
dummy_images_adv = fgsm_attack(model, dummy_images.clone(), dummy_labels, epsilon=0.3)

with torch.no_grad():
    adv_preds = model(dummy_images_adv).argmax(dim=1)

changed = (orig_preds != adv_preds).sum().item()
print(f"  Zmienione predykcje po FGSM (eps=0.3): {changed}/{len(dummy_labels)}")

# === OAI-9B: Multi-label classification ===
print("\n=== Multi-label Classification (wzorzec: III OAI — klasyfikacja wieloetykietowa) ===")

class MultiLabelCNN(nn.Module):
    """CNN z wyjściem Sigmoid dla multi-label (NIE softmax!)."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, num_classes),
            nn.Sigmoid()  # ← SIGMOID, nie Softmax!
        )
    
    def forward(self, x):
        return self.classifier(self.features(x))

# Kluczowa różnica: BCELoss zamiast CrossEntropyLoss
model_ml = MultiLabelCNN(num_classes=5)
criterion = nn.BCELoss()  # Binary Cross Entropy dla multi-label

# Syntetyczne dane multi-label
X = torch.rand(32, 3, 28, 28)
y_multilabel = (torch.rand(32, 5) > 0.5).float()  # Wiele etykiet na raz

pred = model_ml(X)
loss = criterion(pred, y_multilabel)
print(f"  BCELoss: {loss.item():.4f}")

# F1 Macro — metryka z III OAI
from sklearn.metrics import f1_score
y_true = y_multilabel.numpy()
y_pred = (pred.detach().numpy() > 0.5).astype(int)
f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
print(f"  F1 macro (random model): {f1:.4f}")

# === OAI-9C: Denoising Autoencoder ===
print("\n=== Denoising Autoencoder (wzorzec: Rozkład nienormalny, II OAI etap 2) ===")

class DenoisingAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1),  # 28→14
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),  # 14→7
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1),  # 7→14
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 3, stride=2, padding=1, output_padding=1),  # 14→28
            nn.Sigmoid(),
        )
    
    def forward(self, x):
        return self.decoder(self.encoder(x))

# Symulacja: obraz + szum → denoising
dae = DenoisingAutoencoder()
clean = torch.rand(1, 1, 28, 28)
noise_level = 0.3
noisy = torch.clamp(clean + noise_level * torch.randn_like(clean), 0, 1)

# Bez trenowania — tylko demonstracja architektury
with torch.no_grad():
    reconstructed = dae(noisy)

psnr_noisy = 20 * np.log10(1.0 / np.sqrt(nn.MSELoss()(noisy, clean).item()))
psnr_recon = 20 * np.log10(1.0 / np.sqrt(nn.MSELoss()(reconstructed, clean).item()))
print(f"  PSNR zaszumionego: {psnr_noisy:.1f} dB")
print(f"  PSNR po DAE (bez trenowania): {psnr_recon:.1f} dB")
print("  (Po trenowaniu PSNR powinno być znacznie wyższe!)")

print("\n💡 Na II OAI etap 2 trzeba było ODSZUMIĆ obrazy (PSNR) i potem je ZAKLASYFIKOWAĆ (accuracy)!")